# 05 · Classification accuracy (Results 3.5, Table 3, Supplementary Tables S4 and S8, Figure S2)

For each pair of groups and each measure:
* **AUC** in the raw score direction, with a stratified **bootstrap** 95% CI (2,000 resamples) and a two-sided
  **permutation** p-value for AUC ≠ 0.5 (2,000 permutations);
* the **direction-aligned** AUC with its DeLong CI, the **Youden** cut-off (sensitivity, specificity), PPV/NPV at
  5%, 10% and 20% prevalence;
* **repeated cross-validation** (5 folds × 200 repeats): direction and cut-off are learned in the training folds.

Seeds: bootstrap `seed + 10·k + j`, permutation `seed + 100·k + j`, cross-validation `seed + k + j`
(k = contrast, j = measure), so every number can be reproduced exactly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
import sys
sys.path.insert(0, "..")          # dbiat_analysis.py is in the folder above notebooks/
import dbiat_analysis as A

cfg = A.load_settings()
S = cfg["stats"]
LAB = cfg["measures"]
df = pd.read_csv(A.path(cfg, "derived", "scores.csv"))
pd.set_option("display.width", 250, "display.max_columns", 40)

def contrast_data(d, a, b, m):
    d = d[d.group.isin([a, b])]
    return (d.group == b).astype(int).values, d[m].values

## 1. AUC table (all measures, all contrasts)

In [ ]:
rows = []
for k, (a, b) in enumerate(A.CONTRASTS):
    for j, m in enumerate(A.MEASURES):
        y, s = contrast_data(df, a, b, m)
        auc = A.auc_score(y, s)
        assert abs(auc - roc_auc_score(y, s)) < 1e-12
        sign = 1 if auc >= 0.5 else -1
        sa = sign * s
        r = dict(contrast=f"{a} vs {b}", reference=a, positive=b, measure=m,
                 n_ref=int((y == 0).sum()), n_pos=int((y == 1).sum()), auc=auc,
                 direction="higher in positive" if sign == 1 else "lower in positive")
        r["auc_aligned"], r["delong_ci_low"], r["delong_ci_high"] = A.delong_ci(y, sa)
        boot = A.bootstrap_auc(y, s, S["n_bootstrap"], S["random_seed"] + 10 * k + j)
        r["boot_ci_low"], r["boot_ci_high"] = np.percentile(boot, [2.5, 97.5])
        r["p_perm_vs_0.5"] = A.permutation_auc_p(y, s, S["n_permutations"], S["random_seed"] + 100 * k + j)
        yd = A.youden(y, sa)
        r.update(youden_threshold=sign * yd["threshold"], sensitivity=yd["sensitivity"],
                 specificity=yd["specificity"], youden_J=yd["J"])
        for prev in S["prevalences"]:
            ppv, npv = A.ppv_npv(yd["sensitivity"], yd["specificity"], prev)
            r[f"PPV_at_{int(prev * 100)}pct"], r[f"NPV_at_{int(prev * 100)}pct"] = ppv, npv
        r["balanced_accuracy_resub"] = (yd["sensitivity"] + yd["specificity"]) / 2
        r.update(A.cv_classification(y, s, S["cv_folds"], S["cv_repeats"], S["random_seed"] + k + j))
        rows.append(r)
auc = pd.DataFrame(rows)
A.write_table(auc, cfg, "auc_full")
auc[["contrast", "measure", "auc", "boot_ci_low", "boot_ci_high", "p_perm_vs_0.5", "cv_balanced_accuracy"]].round(3)

## 2. Table 3 (AUC [95% bootstrap CI])

In [ ]:
t3 = auc.assign(cell=lambda d: d.apply(lambda r: f"{r.auc:.3f} [{r.boot_ci_low:.2f}, {r.boot_ci_high:.2f}]", axis=1))
table3 = t3.pivot(index="measure", columns="contrast", values="cell").reindex(A.MEASURES)
table3 = table3[[f"{a} vs {b}" for a, b in A.CONTRASTS]]
table3.index = [LAB[m] for m in table3.index]
table3 = table3.reset_index().rename(columns={"index": "Measure"})
A.write_table(table3, cfg, "Table3_AUC")
table3

## 3. Is D_Composite better than the other measures? (DeLong tests on direction-aligned scores)

In [ ]:
rows = []
for a, b in A.CONTRASTS:
    y, s1 = contrast_data(df, a, b, "D_Composite")
    s1a = s1 if A.auc_score(y, s1) >= .5 else -s1
    for m in A.MEASURES:
        if m == "D_Composite":
            continue
        _, s2 = contrast_data(df, a, b, m)
        s2a = s2 if A.auc_score(y, s2) >= .5 else -s2
        rows.append(dict(contrast=f"{a} vs {b}", comparison=f"D_Composite vs {m}", **A.delong_test(y, s1a, s2a)))
delong = pd.DataFrame(rows)
A.write_table(delong, cfg, "delong_Dcomposite_vs_others")
print("Comparisons with p < .05:", int((delong.p < .05).sum()), "of", len(delong))
delong.round(4)

## 4. What an AUC of this size means in practice (Table S4)
PPV and NPV at 5%, 10% and 20% prevalence, (i) for the equal-variance binormal model with sensitivity = specificity
(observed AUC of D_Composite for Control vs Suicidal, and AUCs of 0.70 and 0.80 for comparison),
and (ii) for the observed Youden cut-off of D_Composite (Control vs Suicidal).

In [ ]:
best = auc.query("contrast == 'Control vs Suicidal' and measure == 'D_Composite'").iloc[0]
rows = []
from scipy import stats
for label, a_ in [("Binormal model, sensitivity = specificity (observed AUC)", best.auc),
                  ("Binormal model, sensitivity = specificity (AUC 0.70)", 0.70),
                  ("Binormal model, sensitivity = specificity (AUC 0.80)", 0.80)]:
    se = A.binormal_sens(a_)
    for prev in S["prevalences"]:
        ppv, npv = A.ppv_npv(se, se, prev)
        rows.append(dict(scenario=label, auc=a_, d_prime=np.sqrt(2) * stats.norm.ppf(a_), sensitivity=se,
                         specificity=se, prevalence=prev, PPV=ppv, NPV=npv, false_pos_per_true_pos=(1 - ppv) / ppv))
for prev in S["prevalences"]:
    ppv, npv = A.ppv_npv(best.sensitivity, best.specificity, prev)
    rows.append(dict(scenario="Observed Youden cut-off", auc=best.auc, sensitivity=best.sensitivity,
                     specificity=best.specificity, prevalence=prev, PPV=ppv, NPV=npv,
                     false_pos_per_true_pos=(1 - ppv) / ppv))
ppv_tab = pd.DataFrame(rows)
A.write_table(ppv_tab, cfg, "TableS4_PPV_illustration")
print(f"D_Composite, Control vs Suicidal: AUC = {best.auc:.3f}; Youden sensitivity = {best.sensitivity:.2f}, "
      f"specificity = {best.specificity:.2f}; cross-validated balanced accuracy = {best.cv_balanced_accuracy:.3f} "
      f"[{best.cv_balanced_accuracy_p2_5:.3f}, {best.cv_balanced_accuracy_p97_5:.3f}]")
ppv_tab.round(3)

## 5. Figure S2 · ROC curves (direction-aligned)

In [ ]:
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "pdf.fonttype": 42})
cols = {"D_RT": "#0072B2", "D_ER": "#009E73", "D_Composite": "#D55E00", "life_er": "#CC79A7"}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (a, b) in zip(axes, A.CONTRASTS):
    for m, c in cols.items():
        y, s = contrast_data(df, a, b, m)
        r = auc.query("measure == @m and reference == @a and positive == @b").iloc[0]
        up = r.auc >= .5
        fpr, tpr, _ = roc_curve(y, s if up else -s)
        lo, hi = (r.boot_ci_low, r.boot_ci_high) if up else (1 - r.boot_ci_high, 1 - r.boot_ci_low)
        ax.plot(fpr, tpr, color=c, lw=2,
                label=f"{LAB[m]}: {r.auc_aligned:.3f} [{lo:.2f}, {hi:.2f}]" + ("" if up else " (reversed)"))
    ax.plot([0, 1], [0, 1], ls="--", color="0.6")
    ax.set_title(f"{b} (positive) vs {a}")
    ax.set_xlabel("1 − specificity"); ax.set_ylabel("Sensitivity")
    ax.legend(fontsize=8.5, loc="lower right", title="AUC (aligned) [95% bootstrap CI]", title_fontsize=8.5)
    ax.set_aspect("equal")
fig.tight_layout()
A.save_fig(fig, cfg, "FigS2_ROC_curves", tiff=False)
plt.show()